# 02 · CNN Baseline — a single held-out fold, end to end

**Objective.** Show the *complete* forecasting pipeline on one fold, as the smallest
runnable example of everything the multi-station study does. Everything in notebooks
03–08 is this same machinery repeated over more stations, horizons, and feature sets.

**What this fold is**
| setting | value |
|---|---|
| target | `SWC_20` (soil-water content at 20 cm) |
| inputs | `SWC_20` history only (feature config **A**, the baseline) |
| model | `LargerCNN` — 2×Conv1d (16→32) + MLP head, ~2.2k params |
| input window | 24 h |
| forecast horizon | 72 h ahead |
| train | all station-years **except** the held-out one and the validation year |
| validation | 2019 of the other stations (early stopping only) |
| test | held-out **FD11 / 2023** (never seen in training or early stopping) |

**Implementation.** The whole fold is one call to `src.evaluation.run_fold`, which
chains `fold_keys` → `MinMaxScaler` (fit on train only) → `make_windows` →
`src.training.train_model` (Adam + MSE, early stopping on val) → inverse-transform →
metrics in original SWC units. The CNN itself is `src.models.LargerCNN`.

**Where the original code came from.** The 150 production runs analysed in notebooks
03–08 were produced by the original monolithic notebook, backed up at
`archive/cnn_experiments_original_monolithic.ipynb` and run live on Colab on
2026-11-17. This notebook is the *refactored* version of that same pipeline, now split
into the reusable `src/` package so a single fold is legible and re-runnable.

**How to reproduce.**
* In Colab: Runtime → GPU, run the two cells below (~2–4 min on a T4/L4).
* From a shell: `python main.py --features swc --horizon 72 --stations FD11 --test-years 2023`
  reproduces this fold and writes the row to `results/experiments.csv`.

> ⚠️ **Needs PyTorch + a GPU is recommended.** The outputs saved in this notebook were
> generated on **Google Colab (NVIDIA L4 GPU) on 2026-11-19** — they are a **freshly
> computed** run of the code below, *not* loaded from the November-17 CSVs. They match
> the saved November-17 value for this fold (RMSE ≈ 0.0275, n=8665) to within run-to-run
> training noise, which is the point: the refactored pipeline reproduces the original.

In [1]:
import sys, os
# locate project root (the folder containing 'src/') and work from there
p = os.path.abspath(os.getcwd())
while p != os.path.dirname(p) and not os.path.isdir(os.path.join(p,'src')):
    p = os.path.dirname(p)
sys.path.insert(0, p); os.chdir(p)
print('project root:', p)

project root: /content/soil_moisture_research


In [2]:
from src.data_processing import load_dataframe, build_station_year_store
from src.evaluation import run_fold
from src import config as C
df = load_dataframe()
store = build_station_year_store(df, C.SELECTED_STATIONS)
row, hist = run_fold(store, C.SELECTED_STATIONS, 'FD11', 2023,
                     C.feature_columns('swc'), window=24, horizon=72,
                     epochs=50, patience=8)
print(f"RMSE={row['RMSE']:.4f}  persistence={row['persistence_rmse']:.4f}  "
      f"impr_vs_persist={row['impr_vs_persist_pct']:+.1f}%  "
      f"epochs={row['epochs_completed']} (best {row['best_val_epoch']})")

RMSE=0.0276  persistence=0.0275  impr_vs_persist=-0.3%  epochs=21 (best 13)


**Reading the result.** At a 72-hour horizon the SWC-only CNN is *essentially tied*
with persistence (here RMSE 0.0276 vs 0.0275, i.e. −0.3%). That is the central finding
the rest of the study quantifies at scale: the recent soil-moisture history already
carries almost all of the predictable signal, so the interesting question — the one the
research is actually about — is whether any **exogenous** input (rain, temperature, …)
adds anything *on top of* this baseline. Notebooks 03–07 test exactly that.

Run-to-run note: `epochs`/`best_val_epoch` and the 3rd–4th decimal of RMSE vary slightly
with GPU/seed scheduling; the saved November-17 value for this fold is RMSE 0.02751.